# 11.8 · 序列标注：HMM 与 Viterbi / Sequence Labeling: HMM & Viterbi

> **课程定位 / Where this fits**
> 第 8 课，**Part 11 · 经典 NLP**。
> Lesson 8, **Part 11 · Classic NLP**.
>
> 11.7 的 NER 是序列标注的一种。本课系统讲**序列标注**(以**词性标注 PoS** 为例：给每个词标名词/动词/形容词…)，核心在于**利用标签之间的依赖**——"冠词后面多半是名词"。我们**从零实现隐马尔可夫模型(HMM) + Viterbi 动态规划算法**，在真实的 Penn Treebank 上做词性标注。这套"结构化预测"思想是 CRF、乃至现代序列模型的根基。
> NER (11.7) is one kind of sequence labeling. This lesson covers **sequence labeling** systematically (via **PoS tagging**: label each word noun/verb/adjective…), whose essence is **exploiting dependencies between labels** — "a determiner is usually followed by a noun." We **implement a Hidden Markov Model (HMM) + the Viterbi dynamic-programming algorithm from scratch** for PoS tagging on the real Penn Treebank. This "structured prediction" idea underlies CRFs and modern sequence models.
>
> 💼 **实战/面试视角**："HMM 三要素 / Viterbi 为什么用 DP / 贪心为什么不行 / HMM vs CRF" 是经典考点。
> 💼 **Practical/interview angle:** "HMM's three components / why Viterbi uses DP / why greedy fails / HMM vs CRF" — classic.

> 📐 **符号约定 / Notation**
> - 状态(state)=标签(tag), 观测(observation)=词(word) / states=tags, observations=words
> - 转移 $P(t_i|t_{i-1})$, 发射 $P(w_i|t_i)$, 初始 $P(t_1)$ / transition, emission, initial
> - Viterbi —— 求最可能标签序列的 DP 算法 / DP for the most likely tag sequence

> 💡 **面试相关 / Interview-relevant**
> - "HMM 的三个概率(初始/转移/发射)"（出镜率 ★★★★★）
> - "Viterbi 算法/为什么不能贪心"（★★★★★）
> - "马尔可夫假设是什么"（★★★★）
> - "HMM(生成式) vs CRF(判别式) 区别"（★★★★）
> - "未登录词(OOV)怎么处理"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解序列标注与"标签依赖"为何重要。
   Understand sequence labeling and why label dependencies matter.
2. 掌握 **HMM 三要素**并从数据估计参数。
   Master HMM's three components and estimate them from data.
3. **从零实现 Viterbi**，理解为何用 DP、贪心为何不行。
   Implement Viterbi from scratch; why DP, why greedy fails.
4. 在 Treebank 上评估，对比基线，处理未登录词。
   Evaluate on Treebank vs a baseline; handle unknown words.

## 目录 / TOC
1. [序列标注与词性标注 ⭐](#1)
2. [HMM：三个概率 ⭐](#2)
3. [Viterbi：动态规划解码（从零）⭐](#3)
4. [评估、未登录词与小结 ⭐](#4)


<a id="1"></a>
## 1. 序列标注与词性标注 ⭐ / Sequence Labeling & PoS Tagging

**序列标注**：给一个序列里的**每个元素**打标签，且**标签之间相互依赖**。**词性标注(PoS tagging)** 是最经典的例子——给每个词标它的词性(名词 NN、动词 VB、冠词 DT…)。
**Sequence labeling:** assign a label to **each element** of a sequence, where **labels depend on each other**. **PoS tagging** is the classic example — tag each word with its part of speech (noun NN, verb VB, determiner DT…).

**为什么需要上下文/依赖**：很多词**有歧义**——`book` 在 "read a **book**"(名词)和 "**book** a flight"(动词)里词性不同。光看词本身判不准，得看**上下文和相邻标签**："a/the 后面" 大概率是名词或形容词。
**Why context/dependencies matter:** many words are **ambiguous** — `book` is a noun in "read a **book**" but a verb in "**book** a flight." The word alone is insufficient; we need **context and neighboring labels**: "after a/the" is likely a noun/adjective.

我们用 **Penn Treebank**(nltk 自带，约 3900 句已标注词性)。
We use the **Penn Treebank** (~3900 PoS-tagged sentences in nltk).


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from collections import defaultdict, Counter
import nltk
nltk.download("treebank", quiet=True)
from nltk.corpus import treebank
sns.set_theme(style="whitegrid")

sents = treebank.tagged_sents()
print(f"Treebank: {len(sents)} 句已标注词性")
print("\n一句的词性标注:")
for w, t in sents[0][:10]: print(f"  {w:<14} {t}")
print("  ...(NNP=专有名词, CD=数词, NN=名词, VB=动词, DT=冠词, JJ=形容词, IN=介词)")

# 歧义词: 同一个词有多种词性 / an ambiguous word with multiple tags
wt = defaultdict(Counter)
for s in sents:
    for w, t in s: wt[w.lower()][t] += 1
for w in ["back", "run"]:
    print(f"\n'{w}' 在语料中的词性分布: {dict(wt[w])}  ← 有歧义, 必须看上下文")


<a id="2"></a>
## 2. HMM：三个概率 ⭐ / HMM: Three Probabilities

**隐马尔可夫模型(HMM)** 把词性标注看成：背后有一串**隐藏的词性(状态)**，每个词性"发射"出我们看到的**词(观测)**。它由**三个概率**完全定义(面试必背)：
A **Hidden Markov Model** views tagging as: a hidden chain of **tags (states)**, each "emitting" an observed **word (observation)**. It's fully defined by **three probabilities** (must-know):
1. **初始概率** $P(t_1)$：句子第一个词是某词性的概率。
   **Initial** $P(t_1)$: probability a sentence starts with a given tag.
2. **转移概率** $P(t_i \mid t_{i-1})$：一个词性后面跟另一个词性的概率(如 DT→NN 很高)。这编码了**标签依赖**。
   **Transition** $P(t_i \mid t_{i-1})$: probability one tag follows another (DT→NN is high). This encodes **label dependencies**.
3. **发射概率** $P(w_i \mid t_i)$：某词性"生成"某个词的概率(如 NN→"dog")。
   **Emission** $P(w_i \mid t_i)$: probability a tag emits a given word (NN→"dog").

**马尔可夫假设**：当前词性只依赖**前一个**词性(而非更早的)；当前词只依赖**当前**词性。这是简化(不完全真)，但让模型可解、高效。
**Markov assumption:** the current tag depends only on the **previous** tag; the current word only on the **current** tag. A simplification (not fully true) that makes the model tractable.

参数靠**数数**估计：从训练语料统计各种共现次数再归一化(加平滑避免 0 概率)。
Parameters are estimated by **counting**: tally co-occurrences in training and normalize (with smoothing to avoid zero probabilities).


In [ ]:
np.random.seed(0)
idx = np.random.permutation(len(sents))
train = [sents[i] for i in idx[:3000]]; test = [sents[i] for i in idx[3000:]]

# 未登录词处理: 训练里只出现1次的词→<UNK>, 让模型学到"罕见词"的词性分布 / UNK handling
wc = Counter(w.lower() for s in train for w, _ in s)
vocab = {w for w, c in wc.items() if c >= 2}
def norm(w):
    w = w.lower(); return w if w in vocab else "<UNK>"   # 词表外→UNK / OOV → UNK

tags = sorted({t for s in train for _, t in s}); ti = {t: i for i, t in enumerate(tags)}; T = len(tags)
init = np.ones(T); trans = np.ones((T, T)); emit = [Counter() for _ in range(T)]; tagcnt = np.zeros(T)
for s in train:
    prev = None
    for j, (w, t) in enumerate(s):
        k = ti[t]; tagcnt[k] += 1; emit[k][norm(w)] += 1   # 发射计数 / emission counts
        if j == 0: init[k] += 1                            # 初始计数 / initial counts
        if prev is not None: trans[ti[prev]][k] += 1       # 转移计数 / transition counts
        prev = t
init = init / init.sum(); trans = trans / trans.sum(1, keepdims=True)   # 归一化(已加1平滑) / normalize
Vt = len(vocab) + 1
def emis(k, w): return (emit[k][norm(w)] + 0.01) / (tagcnt[k] + 0.01*Vt)  # 发射概率(加平滑) / emission prob
print(f"训练 {len(train)} 句, {T} 种词性标签, 词表(≥2次) {len(vocab)} 词")

# 可视化转移矩阵的一部分: 哪些词性后面接哪些词性 / transition matrix (subset)
common = ["DT","JJ","NN","NNP","NNS","VB","VBD","VBZ","IN","RB","PRP","CD"]
ci = [ti[t] for t in common if t in ti]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(trans[np.ix_(ci, ci)], annot=True, fmt=".2f", cmap="Blues",
            xticklabels=[tags[i] for i in ci], yticklabels=[tags[i] for i in ci], ax=ax, cbar_kws={"label":"P(下一词性|当前词性)"})
ax.set_xlabel("下一个词性"); ax.set_ylabel("当前词性"); ax.set_title("HMM 转移概率(部分): DT→NN/JJ 很高(冠词后接名词/形容词)")
plt.tight_layout(); plt.show()
print("转移矩阵编码语法规律: DT(冠词)后大概率是 NN(名词)或 JJ(形容词); 这就是'标签依赖'")


<a id="3"></a>
## 3. Viterbi：动态规划解码（从零）⭐ / Viterbi: DP Decoding From Scratch

给定一句话，怎么找**最可能的词性序列**？一句 $n$ 个词、每个词 $T$ 种词性，共 $T^n$ 种可能序列——**穷举不可行**。
Given a sentence, how to find the **most likely tag sequence**? With $n$ words and $T$ tags each, there are $T^n$ possible sequences — **brute force is infeasible**.

**贪心也不行**(面试点)：逐词选当前最可能的词性，可能导致后面无法接续(局部最优≠全局最优)。例如某词单看像名词，但选了它会让整句概率更低。
**Greedy fails too** (interview): picking each word's locally-best tag may not yield the globally best sequence. A word may look like a noun alone, but choosing it lowers the whole sentence's probability.

**Viterbi 算法**用**动态规划**高效求全局最优：从左到右，记录"到第 $j$ 个词、且第 $j$ 个词性为 $t$ 时，最优路径的概率"`dp[j][t]`，并用 `backpointer` 记住是从哪个前驱转移来的。最后从末尾回溯出整条最优路径。复杂度 $O(n T^2)$，可行。
**Viterbi** uses **dynamic programming** for the global optimum: left to right, track `dp[j][t]` = "best path probability ending at word $j$ with tag $t$," plus `backpointers`. Finally backtrack from the end. Complexity $O(n T^2)$ — feasible.

我们在**对数空间**计算(把连乘变连加，避免下溢)。
We compute in **log space** (turning products into sums to avoid underflow).


In [ ]:
logi = np.log(init); logt = np.log(trans)
def viterbi(words):
    n = len(words); dp = np.full((n, T), -1e18); bp = np.zeros((n, T), int)
    for k in range(T): dp[0][k] = logi[k] + np.log(emis(k, words[0]))   # 初始: 初始概率×发射 / init
    for j in range(1, n):
        emis_j = np.array([np.log(emis(k, words[j])) for k in range(T)])  # 当前词对每个词性的发射 / emissions
        M = dp[j-1][:, None] + logt                       # 从每个前驱转移来的得分 (T_prev × T_cur) / scores
        best_prev = M.argmax(0)                           # 每个当前词性的最优前驱 / best predecessor
        dp[j] = M[best_prev, range(T)] + emis_j           # 记录最优得分 / best score
        bp[j] = best_prev                                 # 记住前驱用于回溯 / backpointer
    path = [int(dp[-1].argmax())]                         # 末尾取最优 / best final tag
    for j in range(n-1, 0, -1): path.append(bp[j][path[-1]])   # 回溯 / backtrack
    return [tags[k] for k in reversed(path)]

# 演示: 给一句话解码词性 / decode a sentence
demo = "the company will book strong profits this year".split()
pred = viterbi(demo)
print("Viterbi 解码:")
for w, t in zip(demo, pred): print(f"  {w:<10} {t}")
print("\n注意 'book' 被标成动词(VB/类似), 因为上下文(will ___)→应是动词; 单看词无法判断")
print("Viterbi 用 DP 在 O(n·T²) 内求全局最优标签序列(穷举是 T^n, 贪心会陷局部最优)")


<a id="4"></a>
## 4. 评估、未登录词与小结 ⭐ / Evaluation, Unknown Words & Summary

在测试集上评估 HMM+Viterbi 的**逐词准确率**，并和一个强基线对比：**"每个词都标它在训练里最常见的词性"**(未登录词标成全局最常见词性 NN)。这个基线已经很强(很多词没歧义)，但**HMM 能靠转移概率纠正歧义词**而胜出。
We evaluate HMM+Viterbi's **per-token accuracy** on the test set vs a strong baseline: **"tag each word with its most frequent training tag"** (unknown → globally most frequent, NN). This baseline is already strong (many words are unambiguous), but **HMM wins by resolving ambiguous words via transition probabilities**.


In [ ]:
# 基线: 每词标最常见词性 / baseline: most frequent tag per word
# 注意: 必须只用【训练集】统计(§1 的 wt 用了全语料, 会泄漏测试!) / use TRAIN only (avoid leakage)
wt_train = defaultdict(Counter)
for s in train:
    for w, t in s: wt_train[w.lower()][t] += 1
mft = {w: c.most_common(1)[0][0] for w, c in wt_train.items()}
overall = Counter(t for s in train for _, t in s).most_common(1)[0][0]
def baseline(words): return [mft.get(w.lower(), overall) for w in words]

cor_v = cor_b = tot = 0; cor_vu = tot_u = 0
for sent in test:
    words = [w for w, _ in sent]; gold = [t for _, t in sent]
    pv = viterbi(words); pb = baseline(words)
    for w, g, a, b in zip(words, gold, pv, pb):
        tot += 1; cor_v += (a == g); cor_b += (b == g)
        if w.lower() not in vocab: tot_u += 1; cor_vu += (a == g)   # 未登录词单独统计 / OOV accuracy
acc_v = cor_v/tot; acc_b = cor_b/tot
print(f"HMM+Viterbi 准确率 = {acc_v:.3f}")
print(f"最常见词性基线    = {acc_b:.3f}")
print(f"未登录词(OOV)数 = {tot_u}, HMM 在 OOV 上准确率 = {cor_vu/tot_u:.3f} (靠转移+UNK发射, 比瞎猜强)")
fig, ax = plt.subplots(figsize=(5.5,3.8))
bars = ax.bar(["最常见词性\n基线", "HMM+Viterbi"], [acc_b, acc_v], color=["#bbb","#39c"])
for b,a in zip(bars,[acc_b,acc_v]): ax.text(b.get_x()+b.get_width()/2, a+0.003, f"{a:.3f}", ha="center")
ax.set_ylabel("逐词准确率"); ax.set_ylim(0.8, 0.95); ax.set_title("HMM 用'标签依赖'纠正歧义词 → 胜过逐词基线")
plt.tight_layout(); plt.show()
print("\nHMM 胜在: 利用转移概率(上下文标签)解决歧义词; 基线只看词本身")
print("局限: 马尔可夫假设(只看前一个标签) + 生成式(建模P(词,标签)); 未登录词仍是难点")


```
序列标注: 给每个元素打标签且标签互相依赖; PoS标注是经典例子(词有歧义需上下文)
HMM三要素: 初始P(t1) + 转移P(ti|ti-1)(编码标签依赖) + 发射P(wi|ti); 数数估计+平滑
马尔可夫假设: 当前标签只依赖前一个标签, 当前词只依赖当前标签(简化但可解)
Viterbi: DP 求全局最优标签序列; dp[j][t]+backpointer回溯; O(nT²); 贪心会陷局部最优
未登录词OOV: 训练罕见词→<UNK>学习其词性分布; 仍是难点
HMM(生成式,建模P(w,t)) vs CRF(判别式,建模P(t|w),可用任意特征,通常更好)
现代: BiLSTM-CRF / BERT+CRF; 但HMM/Viterbi思想(结构化解码)仍是根基
```

### 💡 面试速查 / Interview cheat-sheet
1. **HMM 三要素**: 初始 + 转移(标签依赖) + 发射; 数数估计。
   HMM: initial + transition (label dependency) + emission; estimated by counting.
2. **Viterbi**: DP 求最优标签序列, O(nT²); 比穷举(T^n)/贪心好。
   Viterbi: DP for the best tag sequence, O(nT²); beats brute force/greedy.
3. **贪心为何不行**: 局部最优≠全局最优; 需联合优化整条序列。
   Why greedy fails: local ≠ global optimum; must jointly optimize the sequence.
4. **HMM vs CRF**: 生成式P(w,t) vs 判别式P(t|w)(任意特征, 通常更准)。
   HMM vs CRF: generative P(w,t) vs discriminative P(t|w) (arbitrary features, usually better).
5. **OOV**: 罕见词→UNK 学分布; 现代用子词/字符特征。
   OOV: rare→UNK distribution; modern use subword/char features.

### 下一节 / Next
**11.9 文本相似度**——如何衡量两段文本有多像?这是检索、去重、问答匹配、推荐的核心。我们会**从零实现**编辑距离、Jaccard、余弦相似度、以及搜索引擎用的 **BM25**。
**11.9 Text Similarity** — how similar are two texts? Core to retrieval, deduplication, QA matching, recommendation. We'll **implement from scratch** edit distance, Jaccard, cosine similarity, and **BM25** (used by search engines).
